In [8]:
import pandas as pd
import numpy as np
import ast
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
movies = pd.read_csv("/content/tmdb_5000_movies.csv")
credits = pd.read_csv("/content/tmdb_5000_credits.csv")

print("Movies Shape:", movies.shape)
print("Credits Shape:", credits.shape)
print("Movies Columns:\n")
print(movies.columns)

print("\nCredits Columns:\n")
print(credits.columns)

#First 5 Rows
movies.head()
credits.head()

#Dataset Information
movies.info()

#Missing Values
movies.isnull().sum()

#The TMDB 5000 Movie Dataset consists of movie details such as title, genres, overview, keywords, cast, and crew. The **overview** column will be used as the primary text feature for generating content-based recommendations.

Movies Shape: (4803, 20)
Credits Shape: (4803, 4)
Movies Columns:

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')

Credits Columns:

Index(['movie_id', 'title', 'cast', 'crew'], dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 

,0
budget,0
genres,0
homepage,3091
id,0
keywords,0
original_language,0
original_title,0
overview,3
popularity,0
production_companies,0


In [10]:
#Merge the Datasets
movies = movies.merge(credits, on="title")
movies.shape   #to check

movies.columns #to display the columns


Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count', 'movie_id', 'cast', 'crew'],
      dtype='object')

In [11]:
#Task 2: Text Preprocessing for Recommendation

movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]
movies.head()

movies.isnull().sum() #check to handle the missing values
movies['overview'] = movies['overview'].fillna('')
movies = movies.dropna()

movies.isnull().sum()  #checking again

,0
movie_id,0
title,0
overview,0
genres,0
keywords,0
cast,0
crew,0


In [12]:
import ast

def extract_names(text):
    names = []
    for item in ast.literal_eval(text):
        names.append(item['name'])
    return names

movies['genres'] = movies['genres'].apply(extract_names)
movies['genres'].head()

movies['keywords'] = movies['keywords'].apply(extract_names)
movies['keywords'].head()

def extract_cast(text):
    cast = []
    for i, actor in enumerate(ast.literal_eval(text)):
        if i < 3:
            cast.append(actor['name'])
        else:
            break
    return cast
movies['cast'] = movies['cast'].apply(extract_cast)
movies['cast'].head()

def fetch_director(text):
    for member in ast.literal_eval(text):
        if member['job'] == 'Director':
            return [member['name']]
    return []

movies['crew'] = movies['crew'].apply(fetch_director)
movies['crew'].head()

movies['overview'] = movies['overview'].apply(lambda x: x.split())
movies['overview'].head()

def remove_space(items):
    return [item.replace(" ", "") for item in items]

movies['genres'] = movies['genres'].apply(remove_space)
movies['keywords'] = movies['keywords'].apply(remove_space)
movies['cast'] = movies['cast'].apply(remove_space)
movies['crew'] = movies['crew'].apply(remove_space)

In [13]:
movies['tags'] = (movies['overview'] +movies['genres'] +movies['keywords'] +movies['cast'] +movies['crew'])

movies['tags'] = movies['tags'].apply(lambda x: " ".join(x))
movies[['title','tags']].head()

movies['clean_text'] = movies['tags'].str.lower()

import re
movies['clean_text'] = movies['clean_text'].apply(
    lambda x: re.sub(r'[^a-zA-Z\s]', '', x))

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stop_words = set(ENGLISH_STOP_WORDS)
movies['clean_text'] = movies['clean_text'].apply(
    lambda text: " ".join(
        word for word in text.split()
        if word not in stop_words))

movies = movies[['movie_id','title','clean_text']]
movies.head()



,movie_id,title,clean_text
0,19995,Avatar,nd century paraplegic marine dispatched moon p...
1,285,Pirates of the Caribbean: At World's End,captain barbossa long believed dead come life ...
2,206647,Spectre,cryptic message bonds past sends trail uncover...
3,49026,The Dark Knight Rises,following death district attorney harvey dent ...
4,49529,John Carter,john carter warweary military captain whos ine...


In [14]:
#Task 3: TF-IDF Vectorization

from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    stop_words='english'
)

tfidf_matrix = tfidf.fit_transform(movies['clean_text'])
print("TF-IDF Matrix Shape:")
print(tfidf_matrix.shape)

TF-IDF Matrix Shape:
(4809, 5000)


In [15]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(tfidf_matrix)
print(similarity.shape)
similarity[:5, :5]

(4809, 4809)


array([[1.        , 0.04198276, 0.02450586, 0.02057446, 0.09403021],
       [0.04198276, 1.        , 0.01257216, 0.00349036, 0.03455483],
       [0.02450586, 0.01257216, 1.        , 0.00869354, 0.03807973],
       [0.02057446, 0.00349036, 0.00869354, 1.        , 0.01316926],
       [0.09403021, 0.03455483, 0.03807973, 0.01316926, 1.        ]])

In [16]:
#Task 5 Recommendation Function
def recommend(movie_name, top_n=5):
    # Check if movie exists
    if movie_name not in movies['title'].values:
        return ["Movie not found!"]
    # Find movie index
    index = movies[movies['title'] == movie_name].index[0]
    # Similarity scores
    distances = list(enumerate(similarity[index]))
    # Sort by similarity
    distances = sorted(distances,key=lambda x: x[1], reverse=True)
    recommendations = []

    for i in distances[1:top_n+1]:
        recommendations.append(movies.iloc[i[0]].title)

    return recommendations

In [17]:
recommend("Avatar")
recommend("Batman Begins")
recommend("The Dark Knight")
recommend("Iron Man")

['Iron Man 2',
 'Iron Man 3',
 'Avengers: Age of Ultron',
 'Ant-Man',
 'The Avengers']

In [18]:
#saving the model for streamlit aplication

import pickle
pickle.dump(movies, open("movies.pkl", "wb"))
pickle.dump(similarity, open("similarity.pkl", "wb"))


In [19]:
import os

os.listdir()

['.config',
 'movies.pkl',
 'similarity.pkl',
 'tmdb_5000_credits.csv',
 'tmdb_5000_movies.csv',
 'sample_data']